In [98]:
import pandas as pd

df=pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [99]:
# Tokenize
def tokenize(text):
  text=text.lower()

  text=text.replace("'","")
  text=text.replace('?','')

  return text.split()

In [100]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [101]:
vocab={'<UNK>':0}

In [102]:
def build_vocab(row):
  token_question=tokenize(row['question'])
  token_answer=tokenize(row['answer'])

  merged_tokens=token_question+token_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)



In [103]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [104]:
len(vocab)

324

In [105]:
# convert words to numerical indices
def text_to_indices(text,vocab):
  indexed_text=[]
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [106]:
text_to_indices("What is Wscube Tech?",vocab)

[1, 2, 0, 0]

In [107]:
import torch
from  torch.utils.data import Dataset,DataLoader



In [108]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    row=self.df.iloc[index]
    question=text_to_indices(row['question'],self.vocab)
    answer=text_to_indices(row['answer'],self.vocab)

    return torch.tensor(question),torch.tensor(answer)

In [109]:
dataset=QADataset(df,vocab)


In [110]:
dataset[3]

(tensor([ 1,  2,  3, 17, 18, 19, 20, 21, 22]), tensor([23]))

In [111]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [112]:
for question,answer in dataloader:
  print(question)
  print(answer)


tensor([[ 10, 140,   3, 141, 270,  93, 271,   5,   3, 272]])
tensor([[273]])
tensor([[ 10,  75, 208]])
tensor([[209]])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]])
tensor([[173]])
tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]])
tensor([[72]])
tensor([[  1,   2,   3,   4,   5, 109]])
tensor([[317]])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]])
tensor([[179]])
tensor([[10, 11, 12, 13, 14, 15]])
tensor([[16]])
tensor([[10, 75, 76]])
tensor([[77]])
tensor([[ 42, 250, 251, 118, 252, 253]])
tensor([[254]])
tensor([[ 1,  2,  3, 92, 93, 94]])
tensor([[95]])
tensor([[  1,   2,   3,  92, 137,  19,   3,  45]])
tensor([[185]])
tensor([[ 42, 137,   2, 138,  39, 175, 269]])
tensor([[99]])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]])
tensor([[128]])
tensor([[  1,   2,   3,   4,   5, 286]])
tensor([[287]])
tensor([[ 10,  75,   3, 296,  19, 297]])
tensor([[298]])
tensor([[ 10,  96,   3, 104, 239]])
tensor([[240]])
tensor([[ 1,  2,  3, 69,  5, 53]])
tensor([[260]])
tensor([[7

In [113]:
import torch.nn as nn

In [114]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [115]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [116]:
learning_rate=0.001
epochs=20

In [117]:
model=SimpleRNN(len(vocab))


In [118]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [119]:
for epoch in range(epochs):
  total_loss=0
  for question,answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output=model(question)

    # loss
    loss=loss_fn(output,answer[0])

    # backward pass
    loss.backward()

    # update weights
    optimizer.step()

    total_loss+=loss.item()



  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 524.986515
Epoch: 2, Loss: 455.326884
Epoch: 3, Loss: 373.986092
Epoch: 4, Loss: 314.072499
Epoch: 5, Loss: 262.389605
Epoch: 6, Loss: 214.217127
Epoch: 7, Loss: 170.399265
Epoch: 8, Loss: 132.359189
Epoch: 9, Loss: 101.116281
Epoch: 10, Loss: 76.973283
Epoch: 11, Loss: 59.189895
Epoch: 12, Loss: 46.228507
Epoch: 13, Loss: 36.869093
Epoch: 14, Loss: 30.038010
Epoch: 15, Loss: 25.055459
Epoch: 16, Loss: 21.101635
Epoch: 17, Loss: 17.694688
Epoch: 18, Loss: 15.674923
Epoch: 19, Loss: 13.355947
Epoch: 20, Loss: 11.609188


In [125]:
def predict(model,question,thershold=0.5):

  # Convert Question to Number
  question=text_to_indices(question,vocab)

  # tensor
  question_tensor=torch.tensor(question).unsqueeze(0)

  # send to model
  output=model(question_tensor)

  # convert logits to probs
  probs=torch.nn.functional.softmax(output,dim=1)

  # find index of max probs
  value,index=torch.max(probs,dim=1)


  if value<thershold:
    print("I Don't Know")

  print(list(vocab.keys())[index])





In [127]:
predict(model,"What is the boiling point of water in Celsius?")

100


In [129]:
predict(model,"What is the largest planet in our solar system?")

jupiter


In [130]:
list(vocab.keys())[7]

'paris'